# Create a sample Y-junction network with PyVista

This notebook builds a simple 1D metric graph representing a Y-junction as an
unstructured grid with VTK line cells, adds sample vertex and edge data, and saves the
result as a VTK file.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
from pyvista import CellType
from pyvista.core.dataset import ActiveArrayInfo
from pyvista.core.utilities.arrays import FieldAssociation

In [ ]:
# Four vertices: inlet, junction, and two outlets.
points = np.array(
    [
        [0.0, 0.0, 0.0],   # inlet
        [1.0, 0.0, 0.0],   # junction
        [2.0, 0.8, 0.0],   # upper outlet
        [2.0, -0.8, 0.0],  # lower outlet
    ],
    dtype=float,
)

# Three 1D cells: inlet->junction, junction->upper, junction->lower.
# Each cell is encoded as: [number_of_points, point_id_0, point_id_1].
cells = np.array(
    [
        2, 0, 1,
        2, 1, 2,
        2, 1, 3,
    ],
    dtype=np.int64,
)

celltypes = np.array([CellType.LINE, CellType.LINE, CellType.LINE], dtype=np.uint8)
network = pv.UnstructuredGrid(cells, celltypes, points)

# Point data: sample pressure values at vertices.
network.point_data["pressure"] = np.array([100.0, 95.0, 92.0, 90.0])

# Cell data: sample vessel radius on each edge.
network.cell_data["radius"] = np.array([0.30, 0.20, 0.18])

print(f"Number of points: {network.n_points}")
print(f"Number of cells: {network.n_cells}")
print("Point arrays:", list(network.point_data.keys()))
print("Cell arrays:", list(network.cell_data.keys()))

In [ ]:
output_path = Path("y_junction_network.vtk")
network.save(output_path)
print(f"Saved VTK file to: {output_path.resolve()}")
print(network)
print("Point data:", list(network.point_data.keys()))
print("Cell data:", list(network.cell_data.keys()))

In [ ]:
# Optional visualization.
plotter = pv.Plotter()
plotter.add_mesh(
    network,
    scalars="pressure",
    render_lines_as_tubes=True,
    line_width=12,
    cmap="viridis",
    show_scalar_bar=True,
)
plotter.add_point_labels(
    points,
    [f"p={p:.1f}" for p in network.point_data["pressure"]],
    point_size=18,
    font_size=14,
)
plotter.show()